<a href="https://colab.research.google.com/github/prathyusha2020/AAIS_322-Natural-Language-Processing-and-Computer-Vision/blob/main/Module_2_Interactive_Session_2_07_30_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RNN

In [4]:
import numpy as np
np.random.seed(0)

vocab =["I", "love","recurrent", "neural","networks"]
word_to_idx= {w:i for i,w in enumerate(vocab)}
print(word_to_idx)
vocab_size= len(vocab)

#RNN Weights : the three steps
hidden_size =4
W_xh = np.random.randn(hidden_size, vocab_size) * 0.1   # input  -> state
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1  # state  -> state
W_hy = np.random.randn(vocab_size, hidden_size) * 0.1   # state  -> output

def one_hot(word):
    v = np.zeros((vocab_size, 1))
    v[word_to_idx[word]] = 1
    return v

def rnn_step(x, h_prev):
    h = np.tanh(W_hh @ h_prev + W_xh @ x)   # update the state
    y = W_hy @ h                             # produce the output (raw scores)
    return y, h
sentence = ["I", "love", "recurrent", "neural"]
h = np.zeros((hidden_size, 1))


for word in sentence:
    x = one_hot(word)
    y, h = rnn_step(x, h)
    pred_idx = np.argmax(y)
    print(f"input: {word:<10} -> predicted next word: {vocab[pred_idx]:<10} | hidden state: {h.ravel().round(2)}")


{'I': 0, 'love': 1, 'recurrent': 2, 'neural': 3, 'networks': 4}
input: I          -> predicted next word: I          | hidden state: [ 0.17 -0.1   0.01  0.03]
input: love       -> predicted next word: I          | hidden state: [-0.01  0.15  0.16  0.15]
input: recurrent  -> predicted next word: I          | hidden state: [ 0.11 -0.04  0.1  -0.05]
input: neural     -> predicted next word: I          | hidden state: [0.2  0.02 0.02 0.03]


#Home Work Question

## Run this code and see how RNN generate questions and give me your feedback on what u **understood**

In [ ]:
import numpy as np
np.random.seed(1)

sentences = [
    ["I", "love", "recurrent", "neural", "networks"],
    ["I", "love", "deep", "learning", "models"],
]
vocab = sorted(set(w for s in sentences for w in s))
word_to_idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
hidden_size = 16

W_xh = np.random.randn(hidden_size, vocab_size) * 0.1
W_hh = np.random.randn(hidden_size, hidden_size) * 0.1
W_hy = np.random.randn(vocab_size, hidden_size) * 0.1

def one_hot(word):
    v = np.zeros((vocab_size, 1))
    v[word_to_idx[word]] = 1
    return v

def softmax(y):
    e = np.exp(y - np.max(y))
    return e / e.sum()

lr, epochs = 0.1, 1000
for epoch in range(epochs):
    dW_xh, dW_hh, dW_hy = np.zeros_like(W_xh), np.zeros_like(W_hh), np.zeros_like(W_hy)
    total_loss = 0
    for sentence in sentences:
        inputs, targets = sentence[:-1], sentence[1:]
        h = np.zeros((hidden_size, 1))
        xs, hs, ps = {}, {-1: h}, {}
        for t, (in_word, target_word) in enumerate(zip(inputs, targets)):
            xs[t] = one_hot(in_word)
            hs[t] = np.tanh(W_hh @ hs[t-1] + W_xh @ xs[t])
            ps[t] = softmax(W_hy @ hs[t])
            total_loss += -np.log(ps[t][word_to_idx[target_word], 0] + 1e-9)

        dh_next = np.zeros((hidden_size, 1))
        for t in reversed(range(len(inputs))):
            dy = ps[t].copy()
            dy[word_to_idx[targets[t]]] -= 1
            dW_hy += dy @ hs[t].T
            dh = W_hy.T @ dy + dh_next
            dh_raw = (1 - hs[t] ** 2) * dh
            dW_xh += dh_raw @ xs[t].T
            dW_hh += dh_raw @ hs[t-1].T
            dh_next = W_hh.T @ dh_raw

    for dparam in [dW_xh, dW_hh, dW_hy]:
        np.clip(dparam, -5, 5, out=dparam)
    W_xh -= lr * dW_xh
    W_hh -= lr * dW_hh
    W_hy -= lr * dW_hy

def generate(seed_word, max_words=6):
    h = np.zeros((hidden_size, 1))
    word = seed_word
    generated = [word]
    for _ in range(max_words - 1):
        x = one_hot(word)
        h = np.tanh(W_hh @ h + W_xh @ x)
        y = W_hy @ h
        next_word = vocab[np.argmax(softmax(y).ravel())]   # greedy pick
        generated.append(next_word)
        word = next_word          # <-- prediction fed back in as next input
    return generated

for seed in ["I", "recurrent", "deep"]:
    print(f"seed: '{seed}' -> generated: {' '.join(generate(seed, 6))}")

## Run this code and try to observe what happens

In [ ]:
import numpy as np
np.random.seed(0)

vocab = ["the", "cat", "sat", "on", "a", "mat", "and", "then",
         "slept", "all", "day", "long", "near", "warm", "sun"]
word_to_idx = {w: i for i, w in enumerate(vocab)}
vocab_size = len(vocab)
hidden_size = 8

W_xh = np.random.randn(hidden_size, vocab_size) * 0.5
W_hh = np.random.randn(hidden_size, hidden_size) * 0.5
W_hy = np.random.randn(vocab_size, hidden_size) * 0.5

def one_hot(word):
    v = np.zeros((vocab_size, 1))
    v[word_to_idx[word]] = 1
    return v

def softmax(y):
    e = np.exp(y - np.max(y))
    return e / e.sum()

sentence = ["the", "cat", "sat", "on", "a", "mat", "and", "then",
            "slept", "all", "day", "long", "near", "the", "warm",
            "sun", "and", "then", "slept", "day"]
inputs, targets = sentence[:-1], sentence[1:]
T = len(inputs)   # 19 time steps

# forward pass, caching hidden states
h = np.zeros((hidden_size, 1))
xs, hs, ps = {}, {-1: h}, {}
for t, (in_word, target_word) in enumerate(zip(inputs, targets)):
    xs[t] = one_hot(in_word)
    hs[t] = np.tanh(W_hh @ hs[t-1] + W_xh @ xs[t])
    ps[t] = softmax(W_hy @ hs[t])

# inject ONE error signal at the very last step, then watch it travel backward
dy_last = ps[T-1].copy()
dy_last[word_to_idx[targets[T-1]]] -= 1
dh = W_hy.T @ dy_last

print(f"{'step (t)':>10} | {'steps back':>10} | {'gradient norm':>15}")
grad_norms = []
for t in reversed(range(T)):
    dh_raw = (1 - hs[t] ** 2) * dh          # tanh derivative
    norm = np.linalg.norm(dh_raw)
    grad_norms.append(norm)
    print(f"{t:>10} | {T-1-t:>10} | {norm:>15.8f}")
    dh = W_hh.T @ dh_raw                    # step one more position back

